# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiwateNandini/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [7]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("✅ Token loaded successfully")
else:
    print("❌ Token not found")

✅ Token loaded successfully


In [8]:
!pip -q install duckdb datasets huggingface_hub pyarrow

In [9]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("✅ Hugging Face connected successfully!")

✅ Hugging Face connected successfully!


In [10]:
from datasets import load_dataset
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train",
    token=HF_TOKEN
)

print(ds)
print(ds[:5])

Dataset({
    features: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start'],
    num_rows: 104
})
{'client_hash_id': ['client_04660893ae39614a', 'client_05475c07ed21a83a', 'client_06d356715a8ff3b6', 'client_0797ff3a1fc9a6a5', 'client_08a6a72ff48e62c0'], 'is_active': [True, True, True, True, True], 'has_gsc_access': [True, False, True, False, True], 'has_ga4_access': [True, False, True, False, False], 'access_profile': ['gsc_and_ga4', 'no_search_or_analytics_access', 'gsc_and_ga4', 'no_search_or_analytics_access', 'gsc_only'], 'client_created_date': [datetime.date(2026, 4, 15), datetime.date(2026, 4, 1), datetime.date(2026, 3, 23), datetime.date(2025, 5, 26), datetime.date(2025, 5, 26)], 'client_updated_date': [datetime.date(2026, 6, 27), datetime.date(2026, 6, 27), datetime.date(2026, 7, 5), datetime.date(2026, 6, 27), datetime.date(2026, 6, 27)], 'gsc_data_start': [None,

## Unit of analysis + time window

For my Content Refresh Prioritization lane, one row represents the daily performance of one content page for one client.

I will use data from a mid-panel month (2026-03) to build and evaluate features. This avoids using the final month as training data and follows the assignment recommendation.

The goal is to use these observations to support content refresh decisions.

In [11]:
from datasets import load_dataset
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

for i, row in enumerate(ds):
    print(row)
    if i == 2:
        break

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_users': 0, 'ga4_engaged_sessions': 0, 'ga4_total_engagement_sec': 0, 'sessions_organic': 0, 'sessions_direct': 0, 'sessions_referral': 0, 'sessions_social': 0, 'sessions_paid': 0, 'sessions_ai': 0, 'ai_chatgpt': 0, 'ai_perplexity': 0, 'ai_gemini': 0, 'ai_copilot': 0, 'ai_claude': 0, 'ai_meta': 0, 'ai_other': 0, 'scroll_events': 0}
{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_fe8e8155ce1d47a2', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 5, 'gsc_clicks': 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

### Label
- Content page needs refresh (proxy based on declining search performance).

### Context
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4

### Excluded
- AI-specific traffic fields (ai_chatgpt, ai_gemini, ai_claude, etc.) because they are not required for the initial content refresh model.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
from itertools import islice
import pandas as pd

sample = list(islice(ds, 1000))
df = pd.DataFrame(sample)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nDate range:")
print(df["report_date"].min(), "to", df["report_date"].max())

print("\nMissing values:")
print(df.isnull().sum())

Rows: 1000
Columns: 30

Date range:
2025-01-27 to 2025-01-30

Missing values:
report_date                 0
client_hash_id              0
content_hash_id             0
client_has_gsc              0
client_has_ga4              0
gsc_data_available          0
ga4_data_available          0
gsc_impressions             0
gsc_clicks                  0
gsc_sum_position            0
gsc_avg_position            0
ga4_pageviews               0
ga4_sessions                0
ga4_users                   0
ga4_engaged_sessions        0
ga4_total_engagement_sec    0
sessions_organic            0
sessions_direct             0
sessions_referral           0
sessions_social             0
sessions_paid               0
sessions_ai                 0
ai_chatgpt                  0
ai_perplexity               0
ai_gemini                   0
ai_copilot                  0
ai_claude                   0
ai_meta                     0
ai_other                    0
scroll_events               0
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

This dataset is useful for decision support, but it has limitations.

- Some clients have a longer history than others (unbalanced history).
- Some rows contain only Google Search Console data and not GA4 data.
- The dataset is pseudonymized, so client identities cannot be determined.
- The results are observational and should not be interpreted as causal relationships.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.